<a href="https://colab.research.google.com/github/nyasaniboto/arsenal-performance-trend/blob/main/notebooks/01_load_and_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np

seasons = ['1617','1718','1819','1920','2021','2122','2223',
           '2324','2425','2526']

BASE = "https://www.football-data.co.uk/mmz4281/{s}/E0.csv"

frames = []
for s in seasons:
  d = pd.read_csv(BASE.format(s=s), encoding='latin-1')
  d['season'] = f'20{s[:2]}/{s[2:]}'
  frames.append(d)
  print(s, d.shape)

raw = pd.concat(frames, ignore_index=True)
raw.to_csv('raw.csv', index=False)
print(f'\nCombined: {raw.shape}')
print(raw.groupby('season').size())

1617 (380, 66)
1718 (380, 66)
1819 (380, 63)
1920 (380, 107)
2021 (380, 107)
2122 (380, 107)
2223 (380, 107)
2324 (380, 107)
2425 (380, 121)
2526 (380, 133)

Combined: (3800, 182)
season
2016/17    380
2017/18    380
2018/19    380
2019/20    380
2020/21    380
2021/22    380
2022/23    380
2023/24    380
2024/25    380
2025/26    380
dtype: int64


In [9]:
# finds the columns present in every season
common = set.intersection(*[set(f.columns) for f in frames])
print(len(common))
print(sorted(common))

arsenal = raw[(raw['HomeTeam'] == 'Arsenal') | (raw['AwayTeam'] == 'Arsenal')]
arsenal.to_csv('arsenal.csv', index=False)
print(arsenal.groupby("season").size())

35
['AC', 'AF', 'AR', 'AS', 'AST', 'AY', 'AwayTeam', 'B365A', 'B365D', 'B365H', 'BWA', 'BWD', 'BWH', 'Date', 'FTAG', 'FTHG', 'FTR', 'HC', 'HF', 'HR', 'HS', 'HST', 'HTAG', 'HTHG', 'HTR', 'HY', 'HomeTeam', 'PSA', 'PSCA', 'PSCD', 'PSCH', 'PSD', 'PSH', 'Referee', 'season']
season
2016/17    38
2017/18    38
2018/19    38
2019/20    38
2020/21    38
2021/22    38
2022/23    38
2023/24    38
2024/25    38
2025/26    38
dtype: int64


In [10]:
keep = ["season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "HTHG",
        "HTAG", "HTR", "HS", "AS", "HST", "AST", "HC", "AC", "HF", "AF", "HY",
        "AY", "HR", "AR", "B365H", "B365D", "B365A"]
m = raw[keep].copy()

# the files mix dd/mm/yy and dd/mm/yyyy
m["Date"] = pd.to_datetime(m["Date"], dayfirst=True, format="mixed")
print("Unparsed dates:", m["Date"].isna().sum())
print(m.isna().sum()[m.isna().sum() > 0])

m = m.sort_values("Date").reset_index(drop=True)
m["match_id"] = m.index + 1       # lets the two rows of one match be linked later

home = pd.DataFrame({
    "match_id": m["match_id"], "season": m["season"], "date": m["Date"],
    "team": m["HomeTeam"], "opponent": m["AwayTeam"], "venue": "Home",
    "goals_for": m["FTHG"], "goals_against": m["FTAG"],
    "ht_goals_for": m["HTHG"], "ht_goals_against": m["HTAG"],
    "shots_for": m["HS"], "shots_against": m["AS"],
    "sot_for": m["HST"], "sot_against": m["AST"],
    "corners_for": m["HC"], "corners_against": m["AC"],
    "fouls_for": m["HF"], "fouls_against": m["AF"],
    "yellows": m["HY"], "reds": m["HR"],
})

away = pd.DataFrame({
    "match_id": m["match_id"], "season": m["season"], "date": m["Date"],
    "team": m["AwayTeam"], "opponent": m["HomeTeam"], "venue": "Away",
    "goals_for": m["FTAG"], "goals_against": m["FTHG"],
    "ht_goals_for": m["HTAG"], "ht_goals_against": m["HTHG"],
    "shots_for": m["AS"], "shots_against": m["HS"],
    "sot_for": m["AST"], "sot_against": m["HST"],
    "corners_for": m["AC"], "corners_against": m["HC"],
    "fouls_for": m["AF"], "fouls_against": m["HF"],
    "yellows": m["AY"], "reds": m["AR"],
})

Unparsed dates: 0
Series([], dtype: int64)


In [12]:
tm = pd.concat([home,away], ignore_index=True)
tm['points'] = np.where(tm['goals_for'] > tm['goals_against'], 3,
                        np.where(tm['goals_for'] == tm['goals_against'], 1, 0))

In [13]:
print(tm.groupby("match_id")["points"].sum().value_counts())

points
3    2917
2     883
Name: count, dtype: int64


In [14]:
print(len(tm), "rows; expected", 2 * len(m))
print(tm.groupby(["season", "team"]).size().describe())
print(sorted(tm["team"].unique()))
print(tm[tm["team"] == "Arsenal"].groupby("season")["points"].sum())

7600 rows; expected 7600
count    200.0
mean      38.0
std        0.0
min       38.0
25%       38.0
50%       38.0
75%       38.0
max       38.0
dtype: float64
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']
season
2016/17    75
2017/18    63
2018/19    70
2019/20    56
2020/21    61
2021/22    69
2022/23    84
2023/24    89
2024/25    74
2025/26    85
Name: points, dtype: int64


In [15]:
a = tm[(tm["team"] == "Arsenal") & (tm["season"] == "2025/26")]
print((a["points"] == 3).sum(), (a["points"] == 1).sum(), (a["points"] == 0).
      sum(),
      a["goals_for"].sum(), a["goals_against"].sum())

26 7 5 71 27


In [16]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery

PROJECT_ID = "arsenal-points-analysis"
client = bigquery.Client(project=PROJECT_ID)
client.create_dataset("football", exists_ok=True)

out = tm.copy()
out["date"] = out["date"].dt.date
job = client.load_table_from_dataframe(
    out, f"{PROJECT_ID}.football.team_matches",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()
print(client.get_table(f"{PROJECT_ID}.football.team_matches").num_rows)

7600
